<a href="https://colab.research.google.com/github/photominion777/exposure-value-to-light-value/blob/main/raw-meter/RAW_Meter_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Instruction:**
1. Log into your Google account.
2. Press the "Run all" button at the top (the small ▶ triangle). This renders the live slider interface, the ASCII viewfinder scale, and the real-time physical sensor histogram.
3. Choose the external lighting conditions with the lighting slider.
4. Adjust camera parameters: aperture, shutter, ISO and artistic bias.

**Remark:**

Select the standard baseline parameters (Sunny, f/16, 1/125s, ISO 100). Observe how the system evaluates the 18% mid-gray target versus the scene peak, demonstrating why classic light meters force a massive 800% overutilization of linear raw space when trying to meter for pure highlight protection.


In [31]:
# @title LINEAR RAW EXPOSURE METER SIMULATOR
# (PART 1: LOGIC ENGINE)
import math
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Global option lists representing the manufacturer display values
N_OPTIONS = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.5, 2.8, 3.2, 3.5, 4.0, 4.5, 5.0, 5.6, 6.3, 7.1, 8.0, 9.0, 10, 11, 13, 14, 16, 18, 20, 22, 25, 29, 32]
T_OPTIONS = ["1", "1/1.3", "1/1.6", "1/2", "1/2.5", "1/3.2", "1/4", "1/5", "1/6", "1/8", "1/10", "1/13", "1/15", "1/20", "1/25", "1/30", "1/40", "1/50", "1/60", "1/80", "1/100", "1/125", "1/160", "1/200", "1/250", "1/320", "1/400", "1/500", "1/640", "1/800", "1/1000", "1/1250", "1/1600", "1/2000", "1/2500", "1/3200", "1/4000", "1/5000", "1/6400", "1/8000"]
S_OPTIONS = [50, 64, 80, 100, 125, 160, 200, 250, 320, 400, 500, 640, 800, 1000, 1250, 1600, 2000, 2500, 3200, 4000, 5000, 6400, 8000, 10000, 12800, 16000, 20000, 25600, 51200, 102400]
BIT_DEPTH_OPTIONS = [12, 14, 16]

# Container and static label
output_area = widgets.Output()
lighting_text_right = widgets.Label(value="Sunny", layout=widgets.Layout(width='250px', margin='0 0 0 10px'))

def get_lighting_name(lv):
    ranges = [
        (19.5, 22.5, "Industrial Laser / Lab Light"), (18.5, 19.5, "Arc Welding / High-Power LED"),
        (17.0, 18.5, "Studio Flash / Searchlight"), (15.5, 17.0, "Extreme Sun (Snow/Sand)"),
        (14.5, 15.5, "Sunny"), (13.5, 14.5, "Hazy Sun"), (12.5, 13.5, "Bright Overcast"),
        (11.5, 12.5, "Overcast / Cloudy"), (10.5, 11.5, "Deep Shade"), (9.5, 10.5, "Sunset / Sunrise"),
        (8.5, 9.5, "Very Dynamic Twilight"), (7.5, 8.5, "Bright Street Lighting"),
        (6.5, 7.5, "Blue Hour / City Night"), (5.5, 6.5, "Bright Indoor"), (4.5, 5.5, "Standard Indoor"),
        (3.5, 4.5, "Living Room (Evening)"), (2.5, 3.5, "Dim Indoor"), (1.5, 2.5, "Distant Building Lights"),
        (0.5, 1.5, "Very Dim Interior"), (-0.5, 0.5, "Night Sky / City Skyline"),
        (-1.5, -0.5, "Dim Night Street"), (-2.5, -1.5, "Full Moon (Snow)"),
        (-3.5, -2.5, "Full Moon (Landscape)"), (-4.5, -3.5, "Quarter Moon"),
        (-5.5, -4.5, "Crescent Moon"), (-7.5, -5.5, "Starlight Night")
    ]
    for low, high, name in ranges:
        if low <= lv < high: return name
    return "Transition / Mixed Light"

def calculate_and_display_raw(L_input, N, t_str, S, bit_depth, ec_index):
    # 1. Map UI display values to exact mathematical APEX log steps
    idx_N, idx_t, idx_S = N_OPTIONS.index(N), T_OPTIONS.index(t_str), S_OPTIONS.index(S)
    N_exact = 1.0 * (2**(1/6))**(idx_N - 3)
    t_exact = 1.0 * (2**(-1/3))**idx_t
    S_exact = 100.0 * (2**(1/3))**(idx_S - 3)

    L_eff_peak = float(L_input)
    K = 12.5

    # Definition from 1st article: LV_ext = log2(100 * L_eff / K)
    LV_ext = math.log2((100.0 * L_eff_peak) / K) if L_eff_peak > 0 else -5.0
    lighting_text_right.value = get_lighting_name(LV_ext)

    # 2. Setup fixed physical sensor boundaries and APEX log variables
    Y_sat = (2**bit_depth) - 1            # Constant Hardware Wall (Equation 1 & Section 2)
    y_sat = math.log2(Y_sat)               # Constant APEX Ceiling (Section 2)

    av = math.log2(N_exact**2)             # Aperture Value
    tv = math.log2(t_exact)                # Time Value
    sv_raw = math.log2(S_exact / 100.0)    # Sensor Gain (Equation 1, G(S))

    # 3. System Calibration: STRICTLY REFLECTING BOTH WIKI ARTICLES
    # Pure physical hardware substrate constant from Appendix Section 5
    analog_hardware_base = 31.00
    kappa_log2 = analog_hardware_base - math.log2(2**bit_depth - 1)

    # The exact photometric translation term: lb(K/100)
    # Reverses the light meter calibration factor to isolate pure physical L_eff
    photometric_correction = math.log2(K / 100.0)  # Exactly -3.0 stops for K=12.5

    # Your definitive formula: e_peak = lb(kappa) + LV_ext + lb(K/100)
    e_peak = kappa_log2 + LV_ext + photometric_correction

    # Calculate peak signal value as a direct function of gain (Equation 1 & 2)
    y_peak = e_peak - av + tv + sv_raw
    Y_peak_linear = 2**y_peak

    # 4. Process exposure deviations and shift by user Artistic Bias (Equation 4 & 6)
    delta_y_ETTR = y_peak - y_sat
    ec_user = ec_index * (1.0 / 3.0)
    delta_y_Display = delta_y_ETTR - ec_user

    # 5. Build the ASCII viewfinder scale matrix
    scale_header = " -5       -4       -3       -2       -1        0       +1       +2       +3       +4       +5  "
    scale_ticks = []
    for index in range(-15, 16):
        tick_value = index / 3.0
        if abs(delta_y_Display - tick_value) < (1.0 / 6.0) and -5.1 < delta_y_Display < 5.1:
            scale_ticks.append("▲")
        elif index % 3 == 0: scale_ticks.append("|")
        else: scale_ticks.append("·")

    scale_visual = ("◀ " if delta_y_Display < -5.1 else "  ") + "  ".join(scale_ticks) + (" ▶" if delta_y_Display > 5.1 else "  ")
    clipping_ratio = (Y_peak_linear / Y_sat) * 100.0


# @title LINEAR RAW EXPOSURE METER SIMULATOR
# (PART 2: DASHBOARD & UI INTERFACE)
# ===================================================================================================
# [ INTERNAL LINEAR RAW METER SIMULATION TERMINAL ENGINE ]
# ===================================================================================================
    output_text = f"""===================================================================================================
[ INTERNAL LINEAR RAW METER SIMULATION ]
===================================================================================================
Sensor ADC Bit-Depth Resolution    :  {bit_depth}-bit Domain
Absolute Saturation Limit (Y_sat)  :  {Y_sat:,} Digital Numbers (DN) [Constant]
Peak Linear Channel Signal (Y_peak):  {min(int(Y_peak_linear), Y_sat):,} DN {"[🚨 CLIPPED AT HARDWARE WALL]" if Y_peak_linear > Y_sat else ""}
---------------------------------------------------------------------------------------------------
RAW Sensor Log Ceiling (y_sat)     :  {y_sat:.2f} stops [Constant Baseline]
Environmental Light Value (LV_ext) :  {LV_ext:.2f} (Reflects: {lighting_text_right.value})
ADC Range Utilization              :  {clipping_ratio:.1f}%
---------------------------------------------------------------------------------------------------
True RAW ETTR Deviation (Δy_ETTR)  : {delta_y_ETTR:+.2f} stops
User Exposure Override (EC_user)   : {ec_user:+.2f} stops
---------------------------------------------------------------------------------------------------
Displayed RAW Meter Indicator (Δy_Display): {delta_y_Display:+.2f} stops

{scale_header}
{scale_visual}
===================================================================================================
"""
    if Y_peak_linear > Y_sat:
        output_text += f"🚨 CRITICAL HARDWARE CLIPPING: Exceeds sensor capacity by {delta_y_ETTR:.2f} f-stops.\n"
    elif abs(delta_y_ETTR) < 0.01:
        output_text += "✅ OPTIMAL ETTR EXPOSURE: Perfect sensor capacity utilization."
    else:
        output_text += f"⚠️ UNDERUTILIZED DYNAMIC RANGE: Leaving {abs(delta_y_ETTR):.2f} f-stops of safety headroom."

    with output_area:
        clear_output(wait=True)
        display(HTML(f"<pre style='font-family: Courier New; line-height: 1.2; font-size: 14px;'>{output_text}</pre>"))

        # 7. Initialize and render the visual linear RAW histogram (GMM Model)
        plt.figure(figsize=(10, 4))
        x_vals = np.linspace(0, Y_sat * 1.5, 1000)
        sigma = Y_peak_linear * 0.25

        # Gaussian Mixture Model (GMM) as specified in Technical Appendix
        y_hist = np.exp(-((x_vals - Y_peak_linear * 0.8)**2) / (2 * sigma**2)) * 0.7 + \
                 np.exp(-((x_vals - Y_peak_linear * 0.3)**2) / (2 * (sigma * 1.5)**2)) * 0.4 + \
                 np.exp(-((x_vals - Y_peak_linear)**2) / (2 * (sigma * 0.1)**2)) * 0.3

        # Plot valid versus clipped sensor distribution regions
        plt.fill_between(x_vals, 0, y_hist, where=(x_vals <= Y_sat), color='#2ca02c', alpha=0.6, label='Valid RAW Sensor Data')
        if Y_peak_linear > Y_sat:
            plt.fill_between(x_vals, 0, y_hist, where=(x_vals > Y_sat), color='#d62728', alpha=0.6, label='CRITICAL: Clipping (Data Loss)')

        # Draw the absolute hardware ceiling boundary line (Y_sat Wall)
        plt.axvline(x=Y_sat, color='red', linestyle='--', linewidth=2, label=f'Hardware Wall (Y_sat: {Y_sat:,} DN)')

        # Apply titles and labels
        plt.title('VISUAL SENSOR DASHBOARD\nPhysical Signal Distribution on the Linear RAW Sensor', fontsize=12, fontweight='bold', pad=15)
        plt.xlabel('Linear Sensor Signal Level [Digital Numbers (DN)]', fontsize=10, labelpad=8)
        plt.ylabel('Pixel Distribution Frequency', fontsize=10, labelpad=8)
        plt.xlim(0, Y_sat * 1.3)
        plt.ylim(0, 1.1)
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.legend(loc='upper left', frameon=True, shadow=False)

        plt.tight_layout()
        plt.show()

# Setup UI Layout and Interactive Sliders
radiance_options = list()
for index in range(-18, 67):
    l_phys = (2**(index/3.0)) / (100 / 12.5)
    l_phys = round(l_phys, 1) if l_phys >= 10 else round(l_phys, 2)
    if l_phys not in radiance_options: radiance_options.append(l_phys)

slider_layout, style = widgets.Layout(width='450px'), {'description_width': '140px'}
l_slider = widgets.SelectionSlider(options=radiance_options, value=4096.0, description="Scene Radiance:", readout=False, layout=slider_layout, style=style)
n_slider = widgets.SelectionSlider(options=N_OPTIONS, value=16, description="Aperture (N):", layout=slider_layout, style=style)
t_slider = widgets.SelectionSlider(options=T_OPTIONS, value="1/125", description="Shutter (t):", layout=slider_layout, style=style)
s_slider = widgets.SelectionSlider(options=S_OPTIONS, value=100, description="ISO Speed (S):", layout=slider_layout, style=style)
bit_dropdown = widgets.Dropdown(options=BIT_DEPTH_OPTIONS, value=14, description="Sensor Bit-Depth:", layout=widgets.Layout(width='250px'), style=style)
ec_slider = widgets.IntSlider(min=-9, max=9, step=1, value=0, description="Artistic Bias (EC):", readout=False, layout=slider_layout, style=style)

ec_readout = widgets.Label(value="0.00 stops", layout=widgets.Layout(margin='0 0 0 10px'))
def update_ec_label(change): ec_readout.value = f"{change['new']*(1.0/3.0):+.2f} stops"
ec_slider.observe(update_ec_label, names='value')

# Note: The underlying logic mapping remaines intact, only identifiers are unified
ui_controls = widgets.interactive_output(calculate_and_display_raw, {
    'L_input': l_slider, 'N': n_slider, 't_str': t_slider, 'S': s_slider, 'bit_depth': bit_dropdown, 'ec_index': ec_slider
})
display(widgets.VBox([widgets.HBox([l_slider, lighting_text_right]), n_slider, t_slider, s_slider, widgets.HBox([ec_slider, ec_readout]), widgets.HBox([bit_dropdown])]), output_area)


Output()